Sagemaker Multi-Adapter Endpoints Use Case Demonstration
----------------------------------------------------------------------------------------------------------------------------

## Customer
AnyCompany is an early-stage AI startup operating in the LegalTech space, building contract intelligence software for mid-sized legal teams. Operating in stealth mode, AnyCompany was relying on off-the-shelf closed-source models to interpret contracts but wanted to move toward fine-tuning open models on proprietary legal data to build a more industry-aware, differentiated platform. 

## Use Case 
The customer wants to serve multiple clients in the future and explore the option of hosting a single SageMaker endpoint that can cater to multiple use cases or data types. Rather than deploying separate endpoints for each specialized task — such as contract review, legal question answering, and document summarization — the goal is to consolidate all fine-tuned models (LoRA adapters) behind one shared endpoint. 


## Why Multi-Adapter Endpoints

### The Problem with Separate Endpoints

The traditional approach of deploying one endpoint per use case leads to several challenges
that grow worse as the number of use cases increases.

**Cost multiplication.** Each SageMaker endpoint requires its own set of GPU instances.
Most of these instances sit underutilized, as individual use cases rarely sustain peak traffic simultaneously.

**Operational overhead.** Each endpoint requires independent monitoring, scaling policies, deployment pipelines, and maintenance windows. As the number of endpoints grows from 3 to 10 to 50 — one per client or use case — the operational burden becomes unsustainable.

**Scaling inflexibility.** Scaling decisions must be made per endpoint. One use case may experience a traffic spike while others are idle, but there is no way to share capacity across isolated endpoints.

### The Multi-Adapter Advantage

A multi-adapter endpoint solves these problems by hosting one base model and dynamically
loading lightweight LoRA adapters on demand.

**Cost efficiency.** A single endpoint serves all use cases. Instead of 3 endpoints at $21.27/hr, one endpoint at $14.18/hr (2 instances for redundancy) handles the same workload with headroom for additional adapters. The cost per use case decreases as more adapters are added, since each new adapter requires only a few hundred MB of weights rather than a full
model deployment.


In [3]:
import sagemaker
import boto3
import json
import time
from datetime import datetime

# JumpStart imports (now available in sagemaker 2.x)
from sagemaker.jumpstart.estimator import JumpStartEstimator
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    CategoricalParameter,
    HyperparameterTuner
)

# Initialize session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

print(f"✅ SageMaker Version: {sagemaker.__version__}")
print(f"✅ Role: {role}")
print(f"✅ Region: {region}")
print(f"✅ JumpStart Ready!")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
✅ SageMaker Version: 2.219.0
✅ Role: arn:aws:iam::569202655535:role/service-role/AmazonSageMaker-ExecutionRole-20260204T100714
✅ Region: us-east-2
✅ JumpStart Ready!


**Uploading the data to S3**

Please note that the data was generated using AI is for demonstration purposes only. Please replace with the actual data or feel free to leverage the provided dataset.

In [76]:
import sagemaker

bucket = sagemaker.Session().default_bucket()
data_dir = "/home/sagemaker-user/ABII/datasets"

# Upload contract review dataset
!aws s3 cp {data_dir}/contract_review_dataset.json \
    s3://{bucket}/datasets-v2/contract-review/train/contract_review_dataset.json

# Upload legal summary dataset
!aws s3 cp {data_dir}/legal_summary_dataset.json \
    s3://{bucket}/datasets-v2/legal-summary/train/legal_summary_dataset.json

# Upload legal q&a dataset
!aws s3 cp {data_dir}/legal_summary_dataset.json \
    s3://{bucket}/datasets-v2/legal-qa_dataset/train/legal_summary_dataset.json

print("✅ Both datasets uploaded")

upload: datasets/contract_review_dataset.json to s3://sagemaker-us-east-2-569202655535/datasets-v2/contract-review/train/contract_review_dataset.json
upload: datasets/legal_summary_dataset.json to s3://sagemaker-us-east-2-569202655535/datasets-v2/legal-summary/train/legal_summary_dataset.json
✅ Both datasets uploaded


**Verify the data**

In [77]:
print("📁 Contract Review:")
!aws s3 ls s3://{bucket}/datasets-v2/contract-review/train/ --human-readable

print("\n📁 Legal Summary:")
!aws s3 ls s3://{bucket}/datasets-v2/legal-summary/train/ --human-readable

📁 Contract Review:
2026-02-16 12:42:15   35.3 MiB contract_review_dataset.json

📁 Legal Summary:
2026-02-16 12:42:16   35.3 MiB legal_summary_dataset.json


**Training Launch Script**

1. Model Selection (model file) is from jumpstart, provides a faster startup time as no need to download from hugging face
2. Hugging Face Estimator is being leveraged as it comes with pre-installed libraries with PEFT support built in, container optimized for LLM training.


ml.g5.12xlarge (4× A10G GPUs, 24 GB each):

1. 4 GPUs = training data split 4 ways = ~4× faster
2. 24 GB per GPU = comfortable headroom for QLoRA 7B
3. 48 vCPUs = enough for dataloader_num_workers=4 per GPU
4. 192 GB system RAM = plenty for dataset loading
5. Cost-effective: 7/hr  vs  32/hr for P4d
6. NVLink between GPUs = fast gradient synchronization
7. Slightly expensive but less clock time for training



In [ ]:
from sagemaker.huggingface import HuggingFace
from sagemaker.debugger import ProfilerConfig, FrameworkProfile, DetailedProfilingConfig
import sagemaker

role = sagemaker.get_execution_role()
bucket = sagemaker.Session().default_bucket()
model_uri = "s3://jumpstart-cache-prod-us-east-2/huggingface-training/train-huggingface-llm-mistral-7b.tar.gz"

WANDB_API_KEY = "YOUR KEY"  # ← Replace with your actual key

est_qa_v2 = HuggingFace(
    entry_point="train_lora_v2.py",
    source_dir=".",
    instance_type="ml.g5.12xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    distribution={
        "torch_distributed": {"enabled": True}
    },

    # ─── W&B Environment ───
    environment={
        "WANDB_API_KEY": WANDB_API_KEY,
        "WANDB_PROJECT": "legal-multi-adapter1",
        "WANDB_LOG_MODEL": "false",
        "WANDB_WATCH": "gradients",        # Log gradient histograms
    },
            )
        ),
    ),
)

est_qa_v2.fit(
    {
        "model": model_uri,
        "training": f"s3://{bucket}/datasets-v2/legal-qa/train/",
    },
    wait=True,
)

print(f"\n✅ V2 Training complete! Adapter: {est_qa_v2.model_data}")

See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-00-32-55-162


2026-02-16 00:33:12 Starting - Starting the training job...
2026-02-16 00:33:41 Starting - Preparing the instances for training...
2026-02-16 00:34:11 Downloading - Downloading input data........................
2026-02-16 00:37:48 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
2026-02-16 00:38:05,382 sagemaker-training-toolkit INFO     

In [73]:
# Step 1: Check if training completed
import boto3
sm = boto3.client("sagemaker", region_name="us-east-2")

# Find your job name (from the W&B screenshot or SageMaker console)
job_name = "huggingface-pytorch-training-2026-02-16-00-32-55-162"

status = sm.describe_training_job(TrainingJobName=job_name)
print(f"Status: {status['TrainingJobStatus']}")
print(f"Adapter: {status['ModelArtifacts']['S3ModelArtifacts']}")

Status: Completed
Adapter: s3://sagemaker-us-east-2-569202655535/huggingface-pytorch-training-2026-02-16-00-32-55-162/output/model.tar.gz


**Evaluation Job for Legal Q&A adapter**

Evaluation pipeline encompasses the following steps:

1. Generating the predictions

  - STEP 1: Load base Mistral-7B model (4-bit quantized)
  - STEP 2: Load your QLoRA adapter on top
  - STEP 3: Load test dataset                                   
  - STEP 4: Generate predictions for each test example
  - STEP 5: Save predictions as JSON for review                                                                             
INPUT:  Base model + Adapter + Test questions               
OUTPUT: eval_predictions.json (predicted answers)

2. Using LLM as a judge to evaluate on accuracy, completion and hallucination. Model chosen is Haiku but the recommendation is to choose the latest model that is availiable to you. 

In [ ]:
from sagemaker.huggingface import HuggingFace
import sagemaker

role = sagemaker.get_execution_role()
bucket = sagemaker.Session().default_bucket()
model_uri = "s3://jumpstart-cache-prod-us-east-2/huggingface-training/train-huggingface-llm-mistral-7b.tar.gz"

eval_job_v2 = HuggingFace(
    entry_point="eval_lora.py",
    source_dir=".",
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
)

eval_job_v2.fit(
    {
        "model": model_uri,
        "adapter": adapter_v2_uri,
        "test": f"s3://{bucket}/datasets-v2/legal-qa/test/",
    },
    wait=True,
)

print(f"✅ Eval complete: {eval_job_v2.model_data}")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-11-59-32-041


2026-02-16 11:59:53 Starting - Starting the training job......
2026-02-16 12:00:52 Downloading - Downloading input data...................

In [79]:
# Download the predictions
!mkdir -p /tmp/eval-results-v2
!aws s3 cp {eval_job_v2.model_data} /tmp/eval-output-v2.tar.gz
!tar -xzf /tmp/eval-output-v2.tar.gz -C /tmp/eval-results-v2

import json
with open("/tmp/eval-results-v2/eval_predictions.json") as f:
    predictions = json.load(f)

print(f"✅ Loaded {len(predictions)} V2 predictions")

download: s3://sagemaker-us-east-2-569202655535/huggingface-pytorch-training-2026-02-16-11-59-32-041/output/model.tar.gz to ../../../tmp/eval-output-v2.tar.gz
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.creationtime'
✅ Loaded 45 V2 predictions


In [80]:
# Running the evaluation with LLM as a judge
import boto3, re, json
import numpy as np

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
ACCURACY_THRESHOLD = 0.85 #this was based on the customer requirements, feel free to change it
PASS_SCORE = 4 #this was based on the customer requirements, feel free to change it

JUDGE_PROMPT = """You are a senior legal expert performing a STRICT accuracy evaluation.

**Question:** {instruction}
{context_block}

**Reference Answer (verified correct):** {expected}

**Key Facts That MUST Be Present:** {key_facts}

**Model's Answer Being Evaluated:** {predicted}

EVALUATE strictly:

1. **legal_correctness** (1-5): Are ALL legal concepts, statutes, citations correct?
   5=Everything accurate, 4=Minor omissions nothing wrong, 3=Mostly correct small errors, 2=Significant errors, 1=Fundamentally wrong

2. **completeness** (1-5): Does it cover the key facts listed?
   5=All covered, 4=Most(>75%), 3=About half, 2=Less than half, 1=Almost nothing

3. **hallucination_free** (1-5): Any fabricated legal information?
   5=Zero fabrications, 4=Very minor, 3=Some unsupported, 2=Fabricated citations, 1=Severe

Return ONLY valid JSON:
{{
    "legal_correctness": <1-5>,
    "completeness": <1-5>,
    "hallucination_free": <1-5>,
    "key_facts_covered": ["fact1", "fact2"],
    "key_facts_missed": ["fact3"],
    "errors_found": [],
    "reasoning": "Brief explanation"
}}"""

def judge_answer(pred):
    context_block = f"\n**Context:** {pred['context']}" if pred.get("context") else ""
    prompt = JUDGE_PROMPT.format(
        instruction=pred["instruction"],
        context_block=context_block,
        expected=pred["expected_output"],
        predicted=pred["predicted"],
        key_facts=json.dumps(pred.get("key_facts", [])),
    )
    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-haiku-20240307-v1:0",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 800,
            "temperature": 0,
            "messages": [{"role": "user", "content": prompt}],
        }),
    )
    result = json.loads(response["body"].read())
    text = result["content"][0]["text"]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return None

# Run Judge
print("Running LLM Judge on V2...\n")
for i, pred in enumerate(predictions):
    judgment = judge_answer(pred)
    if judgment is None:
        judgment = {
            "legal_correctness": 0, "completeness": 0, "hallucination_free": 0,
            "key_facts_covered": [], "key_facts_missed": pred.get("key_facts", []),
            "errors_found": ["Parse failure"], "reasoning": "Parse failure"
        }

    is_accurate = (
        judgment.get("legal_correctness", 0) >= PASS_SCORE and
        judgment.get("completeness", 0) >= PASS_SCORE and
        judgment.get("hallucination_free", 0) >= PASS_SCORE
    )
    predictions[i]["judgment"] = judgment
    predictions[i]["is_accurate"] = is_accurate

    status = "✅ PASS" if is_accurate else "❌ FAIL"
    lc = judgment.get("legal_correctness", "?")
    co = judgment.get("completeness", "?")
    hf = judgment.get("hallucination_free", "?")
    print(f"  [{i+1}/{len(predictions)}] {status} | Correct={lc} Complete={co} NoHalluc={hf} | {pred['id']}")
    if not is_accurate:
        print(f"    Missed: {judgment.get('key_facts_missed', [])}")

Running LLM Judge on V2...

  [1/45] ❌ FAIL | Correct=2 Complete=3 NoHalluc=3 | qa_001
    Missed: ['willfulness must be proven beyond reasonable doubt']
  [2/45] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | qa_002
  [3/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | qa_003
  [4/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | qa_004
  [5/45] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | qa_005
  [6/45] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | qa_006
  [7/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | qa_007
  [8/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | qa_008
  [9/45] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | qa_009
  [10/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | qa_010
  [11/45] ❌ FAIL | Correct=3 Complete=3 NoHalluc=4 | qa_011
    Missed: ['no IP infringement carve-out', 'no confidentiality breach carve-out', 'no gross negligence carve-out', 'conflict with indemnification', 'jurisdictional enforceability limits']
  [12/45] ✅ PASS | Correct=4 Complete=4 NoHalluc=4 | qa_012
  [1

**Training for other two adapters**

In [84]:
from sagemaker.huggingface import HuggingFace
import sagemaker

role = sagemaker.get_execution_role()
bucket = sagemaker.Session().default_bucket()
model_uri = "s3://jumpstart-cache-prod-us-east-2/huggingface-training/train-huggingface-llm-mistral-7b.tar.gz"
WANDB_API_KEY = "Replace with your key"

# ═══════════════════════════════════════════════════════
#  ADAPTER 2: CONTRACT REVIEW
#  18,621 examples | max_length=512 | batch=16
#  99th percentile: 526 tokens → 512 covers 98.4%
#  Steps: 18,621 ÷ (16×8) = 145/epoch × 3 = 435
#  Estimated: ~35 min total
# ═══════════════════════════════════════════════════════
est_contract = HuggingFace(
    entry_point="train_lora_adapter.py",
    source_dir=".",
    instance_type="ml.g5.48xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    distribution={"torch_distributed": {"enabled": True}},
    environment={
        "WANDB_API_KEY": WANDB_API_KEY,
        "WANDB_PROJECT": "legal-multi-adapter1",
        "WANDB_RUN_NAME": "contract-review-v1-512-r32-3ep",
        "WANDB_LOG_MODEL": "false",
        "MAX_LENGTH": "512",
        "BATCH_SIZE": "16",
        "TOKENIZERS_PARALLELISM": "false",
    },
)

# ═══════════════════════════════════════════════════════
#  ADAPTER 3: LEGAL SUMMARY
#  17,880 examples | max_length=768 | batch=10
#  99th percentile: 755 tokens → 768 covers 99.3%
#  Steps: 17,880 ÷ (10×8) = 224/epoch × 3 = 671
#  Estimated: ~45 min total
# ═══════════════════════════════════════════════════════
est_summary = HuggingFace(
    entry_point="train_lora_adapter.py",
    source_dir=".",
    instance_type="ml.g5.48xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    distribution={"torch_distributed": {"enabled": True}},
    environment={
        "WANDB_API_KEY": WANDB_API_KEY,
        "WANDB_PROJECT": "legal-multi-adapter1",
        "WANDB_RUN_NAME": "legal-summary-v1-768-r32-3ep",
        "WANDB_LOG_MODEL": "false",
        "MAX_LENGTH": "768",
        "BATCH_SIZE": "10",
        "TOKENIZERS_PARALLELISM": "false",
    },
)

# ═══════════════════════════════════════════════════════
#  LAUNCH BOTH — wait=False for parallel execution
# ═══════════════════════════════════════════════════════
est_contract.fit(
    {
        "model": model_uri,
        "training": f"s3://{bucket}/datasets-v2/contract-review/train/",
    },
    wait=False,
)
print(f"🚀 Contract Review launched: {est_contract.latest_training_job.name}")

est_summary.fit(
    {
        "model": model_uri,
        "training": f"s3://{bucket}/datasets-v2/legal-summary/train/",
    },
    wait=False,
)
print(f"🚀 Legal Summary launched: {est_summary.latest_training_job.name}")

print(f"\n✅ Both jobs running in parallel!")
print(f"📊 Monitor at: https://wandb.ai → legal-multi-adapter")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-12-52-34-311
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


🚀 Contract Review launched: huggingface-pytorch-training-2026-02-16-12-52-34-311


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-12-52-55-370


🚀 Legal Summary launched: huggingface-pytorch-training-2026-02-16-12-52-55-370

✅ Both jobs running in parallel!
📊 Monitor at: https://wandb.ai → legal-multi-adapter

Expected completion:
  Contract Review: ~35 min
  Legal Summary:   ~45 min


**Evaluation for other two adapters**

In [89]:
from sagemaker.huggingface import HuggingFace
import sagemaker

role = sagemaker.get_execution_role()
bucket = sagemaker.Session().default_bucket()
model_uri = "s3://jumpstart-cache-prod-us-east-2/huggingface-training/train-huggingface-llm-mistral-7b.tar.gz"

# Get adapter URIs (paste from Step 1 output)
# adapter_contract_uri = "s3://..."
# adapter_summary_uri = "s3://..."

# ── Eval: Contract Review ──
eval_contract = HuggingFace(
    entry_point="eval_adapter.py",
    source_dir=".",
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    environment={"ADAPTER_TYPE": "contract_review"},
)

# ── Eval: Legal Summary ──
eval_summary = HuggingFace(
    entry_point="eval_adapter.py",
    source_dir=".",
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    environment={"ADAPTER_TYPE": "legal_summary"},
)

# Launch both
eval_contract.fit(
    {"model": model_uri, "adapter": adapter_contract_uri},
    wait=False,
)
print(f"🚀 Contract Review eval: {eval_contract.latest_training_job.name}")

eval_summary.fit(
    {"model": model_uri, "adapter": adapter_summary_uri},
    wait=False,
)
print(f"🚀 Legal Summary eval: {eval_summary.latest_training_job.name}")

print(f"\n✅ Both evals running! ~20 min each")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-15-37-01-608
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


🚀 Contract Review eval: huggingface-pytorch-training-2026-02-16-15-37-01-608


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-02-16-15-37-22-634


🚀 Legal Summary eval: huggingface-pytorch-training-2026-02-16-15-37-22-634

✅ Both evals running! ~20 min each


In [91]:
# Download Contract Review results
!mkdir -p /tmp/eval-contract
!aws s3 cp {eval_contract.model_data} /tmp/eval-contract.tar.gz
!tar -xzf /tmp/eval-contract.tar.gz -C /tmp/eval-contract

# Download Legal Summary results
!mkdir -p /tmp/eval-summary
!aws s3 cp {eval_summary.model_data} /tmp/eval-summary.tar.gz
!tar -xzf /tmp/eval-summary.tar.gz -C /tmp/eval-summary

import json

with open("/tmp/eval-contract/eval_predictions.json") as f:
    contract_predictions = json.load(f)

with open("/tmp/eval-summary/eval_predictions.json") as f:
    summary_predictions = json.load(f)

print(f"✅ Contract Review: {len(contract_predictions)} predictions")
print(f"✅ Legal Summary:   {len(summary_predictions)} predictions")

download: s3://sagemaker-us-east-2-569202655535/huggingface-pytorch-training-2026-02-16-15-37-01-608/output/model.tar.gz to ../../../tmp/eval-contract.tar.gz
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.creationtime'
download: s3://sagemaker-us-east-2-569202655535/huggingface-pytorch-training-2026-02-16-15-37-22-634/output/model.tar.gz to ../../../tmp/eval-summary.tar.gz
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.creationtime'
✅ Contract Review: 10 predictions
✅ Legal Summary:   10 predictions


In [92]:
import boto3, re, json
import numpy as np

#Accuracy Threshold can be setup based on your use case. 
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
ACCURACY_THRESHOLD = 0.85
PASS_SCORE = 4

JUDGE_PROMPT = """You are a senior legal expert performing a STRICT accuracy evaluation.

**Question:** {instruction}
{context_block}

**Reference Answer (verified correct):** {expected}

**Key Facts That MUST Be Present:** {key_facts}

**Model's Answer Being Evaluated:** {predicted}

EVALUATE strictly:

1. **legal_correctness** (1-5): Are ALL legal concepts, statutes, citations correct?
   5=Everything accurate, 4=Minor omissions nothing wrong, 3=Mostly correct small errors, 2=Significant errors, 1=Fundamentally wrong

2. **completeness** (1-5): Does it cover the key facts listed?
   5=All covered, 4=Most(>75%), 3=About half, 2=Less than half, 1=Almost nothing

3. **hallucination_free** (1-5): Any fabricated legal information?
   5=Zero fabrications, 4=Very minor, 3=Some unsupported, 2=Fabricated citations, 1=Severe

Return ONLY valid JSON:
{{
    "legal_correctness": <1-5>,
    "completeness": <1-5>,
    "hallucination_free": <1-5>,
    "key_facts_covered": ["fact1", "fact2"],
    "key_facts_missed": ["fact3"],
    "errors_found": [],
    "reasoning": "Brief explanation"
}}"""

def judge_answer(pred):
    context_block = f"\n**Context:** {pred['context']}" if pred.get("context") else ""
    prompt = JUDGE_PROMPT.format(
        instruction=pred["instruction"],
        context_block=context_block,
        expected=pred["expected_output"],
        predicted=pred["predicted"],
        key_facts=json.dumps(pred.get("key_facts", [])),
    )
    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-haiku-20240307-v1:0",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 800,
            "temperature": 0,
            "messages": [{"role": "user", "content": prompt}],
        }),
    )
    result = json.loads(response["body"].read())
    text = result["content"][0]["text"]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return None


def run_judge_and_report(predictions, adapter_name):
    print(f"\n{'='*60}")
    print(f"  Running LLM Judge: {adapter_name}")
    print(f"{'='*60}\n")

    for i, pred in enumerate(predictions):
        judgment = judge_answer(pred)
        if judgment is None:
            judgment = {
                "legal_correctness": 0, "completeness": 0, "hallucination_free": 0,
                "key_facts_covered": [], "key_facts_missed": pred.get("key_facts", []),
                "errors_found": ["Parse failure"], "reasoning": "Parse failure"
            }
        is_accurate = (
            judgment.get("legal_correctness", 0) >= PASS_SCORE and
            judgment.get("completeness", 0) >= PASS_SCORE and
            judgment.get("hallucination_free", 0) >= PASS_SCORE
        )
        predictions[i]["judgment"] = judgment
        predictions[i]["is_accurate"] = is_accurate

        status = "✅ PASS" if is_accurate else "❌ FAIL"
        lc = judgment.get("legal_correctness", "?")
        co = judgment.get("completeness", "?")
        hf = judgment.get("hallucination_free", "?")
        print(f"  [{i+1}/{len(predictions)}] {status} | Correct={lc} Complete={co} NoHalluc={hf} | {pred['id']}")
        if not is_accurate:
            print(f"    Missed: {judgment.get('key_facts_missed', [])}")

    # Report
    total = len(predictions)
    accurate = sum(1 for p in predictions if p["is_accurate"])
    accuracy = accurate / total

    print(f"\n{'='*60}")
    print(f"  📊 {adapter_name} ACCURACY REPORT")
    print(f"{'='*60}")
    print(f"  Overall:  {accurate}/{total} = {accuracy*100:.1f}%")
    print(f"  Required: {ACCURACY_THRESHOLD*100:.0f}%")
    #print(f"  Status:   {'✅ PASSED' if accuracy >= ACCURACY_THRESHOLD else '❌ FAILED'}")

    print(f"\n  Per-Difficulty:")
    for diff in ["easy", "medium", "hard"]:
        d = [p for p in predictions if p["difficulty"] == diff]
        if d:
            diff_acc = sum(1 for p in d if p["is_accurate"]) / len(d) * 100
            status = "✅" if diff_acc >= 85 else "❌"
            print(f"    {status} {diff:10s}: {diff_acc:.1f}%")

    print(f"\n  Average Scores:")
    for metric in ["legal_correctness", "completeness", "hallucination_free"]:
        vals = [p["judgment"].get(metric, 0) for p in predictions]
        print(f"    {metric:25s}: {np.mean(vals):.2f}")

    return accuracy


# ── Run Judge on Both ──
contract_accuracy = run_judge_and_report(contract_predictions, "CONTRACT REVIEW")
summary_accuracy = run_judge_and_report(summary_predictions, "LEGAL SUMMARY")

# ── Final Summary ──
print(f"\n{'='*60}")
print(f"  📊 ALL THREE ADAPTERS SUMMARY")
print(f"{'='*60}")
print(f"  Legal QA (V2):       77.8%  {'✅' if 0.778 >= 0.85 else '❌'}")
#print(f"  Contract Review:     {contract_accuracy*100:.1f}%  {'✅' if contract_accuracy >= 0.85 else '❌'}")
#print(f"  Legal Summary:       {summary_accuracy*100:.1f}%  {'✅' if summary_accuracy >= 0.85 else '❌'}")
print(f"{'='*60}")


  Running LLM Judge: CONTRACT REVIEW

  [1/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_001
  [2/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_002
  [3/10] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | cr_003
  [4/10] ✅ PASS | Correct=5 Complete=5 NoHalluc=5 | cr_004
  [5/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_005
  [6/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_006
  [7/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_007
  [8/10] ❌ FAIL | Correct=3 Complete=4 NoHalluc=4 | cr_008
    Missed: ['no provision for emergency or injunctive relief from courts']
  [9/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_009
  [10/10] ✅ PASS | Correct=4 Complete=4 NoHalluc=5 | cr_010

  📊 CONTRACT REVIEW ACCURACY REPORT
  Overall:  9/10 = 90.0%
  Required: 85%
  Status:   ✅ PASSED

  Per-Difficulty:
    ❌ easy      : 66.7%
    ✅ medium    : 100.0%
    ✅ hard      : 100.0%

  Average Scores:
    legal_correctness        : 4.10
    completeness             : 4.20
    halluci

**Inference**

**Serving Framework: SageMaker Large Model Inference (LMI)**

SageMaker LMI is AWS's optimized serving container purpose-built for large language model inference. Rather than building a custom serving stack, LMI provides a production-ready container that wraps vLLM in SageMaker infrastructure, giving us enterprise-grade model serving with minimal configuration.

LMI provides the following capabilities out of the box:

1. **Engine Support.** LMI supports multiple inference backends including vLLM, TensorRT-LLM, and DeepSpeed. For this deployment, we use the LMI-Dist engine (based on vLLM), which provides the best balance of performance, LoRA support, and operational simplicity for our use case.

2. **Paged Attention.** Traditional attention mechanisms pre-allocate large contiguous blocks of GPU memory for the KV cache, leading to significant memory waste from fragmentation and over-provisioning. Paged Attention, borrowed from virtual memory concepts in operating systems, allocates KV cache memory in small non-contiguous blocks (pages) on demand. This
eliminates memory waste, allows the engine to serve more concurrent requests within the same GPU memory budget, and prevents out-of-memory errors that would otherwise occur under variable-length workloads.

3. **Continuous Batching.** In traditional static batching, the engine waits for an entire batch to complete before accepting new requests. This means the GPU sits idle while waiting for the longest sequence in the batch to finish generating. Continuous batching solves this by allowing new requests to join mid-generation as slots free up. When a shorter request in
the batch finishes, its slot is immediately filled by the next request in the queue. The GPU never sits idle waiting, resulting in significantly higher throughput. 

4. **Multi-LoRA Adapter Support.** LMI natively supports serving multiple LoRA adapters from a single base model. The base model weights remain fixed in GPU memory while lightweight adapter weights are loaded and swapped per request. Our configuration holds up to 30 adapters on GPU simultaneously with an additional 70 cached on CPU for fast swapping. When a request arrives specifying an adapter that is cached on CPU, the engine transparently moves it to GPU — a process that takes milliseconds due to the small size of LoRA weights (typically 100-400MB compared to 14GB for the full model).

In summary, LMI is vLLM wrapped in SageMaker infrastructure — we get the raw performance of vLLM's optimized inference engine combined with SageMaker's managed scaling, monitoring, endpoint management, and IAM-based access control.

**Inference Components**

Inference Components are the deployment units within a SageMaker endpoint that define what is being served and how resources are allocated.

In this architecture, there are two types of Inference Components:

1. **Base Inference Component.** This is the primary container that loads the full Mistral-7B base model onto GPU, initializes the LMI-Dist engine, and handles all inference requests. It defines the compute resource requirements — how many GPUs are needed, how much memory to allocate, and how many copies to run across instances. The base IC is the single entry point for all invocations when using Method 2 (Base IC + Payload Routing).

2. **Adapter Inference Components.** These are lightweight components that register LoRA adapters with the engine. Each adapter IC points to an S3 location containing the trained adapter weights (adapter_model.safetensors and adapter_config.json). When an adapter IC is created, the engine downloads and registers the adapter, making it available for inference. Adapter ICs do not consume GPU resources independently — they attach to the base IC and share its compute allocation.



**STEP 1: CREATE MODEL**

Registers the base foundation model (Mistral-7B) container image and model artifacts in SageMaker, establishing the foundational model that will be shared across all adapter inference components without duplicating the base model weights in memory.

In [14]:
import boto3
import sagemaker
import json
import time

sm_client = boto3.client("sagemaker")
smr_client = boto3.client("sagemaker-runtime")
role = "arn:aws:iam::569202655535:role/service-role/AmazonSageMaker-ExecutionRole-20260204T100714"
region = "us-east-2"

# ============================================================
# STEP 1: CREATE MODEL
# ============================================================
model_name = "mistral-7b-lmi-clean1"

sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": "763104351884.dkr.ecr.us-east-2.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124",
        "Environment": {
            "HF_MODEL_ID": "s3://sagemaker-us-east-2-569202655535/mistral-tgi-multi-adapter",
            "OPTION_DTYPE": "fp16",
            "OPTION_ENABLE_LORA": "true",
            "OPTION_ENGINE": "MPI",
            "OPTION_MAX_CPU_LORAS": "70",
            "OPTION_MAX_LORAS": "30",
            "OPTION_MAX_LORA_RANK": "32",
            "OPTION_MAX_MODEL_LEN": "6000",
            "OPTION_MAX_ROLLING_BATCH_SIZE": "16",
            "OPTION_ROLLING_BATCH": "lmi-dist",
            "OPTION_TENSOR_PARALLEL_DEGREE": "4",
        },
    },
    ExecutionRoleArn=role,
)
print(f"✅ Model created: {model_name}")

✅ Model created: mistral-7b-lmi-clean1


**STEP 2: CREATE ENDPOINT CONFIG**

Defines the infrastructure specifications (ml.g5.12xlarge instance type, instance count of 2+ for high availability across AZs) and production variants that will host the multi-adapter endpoint, enabling the capacity planning needed for 5,000-10,000 contracts/day workload.

In [16]:
# ============================================================
# STEP 2: CREATE ENDPOINT CONFIG
# ============================================================
endpoint_config_name = "legal-lmi-multi-adapter-epc"

sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "InstanceType": "ml.g5.12xlarge",
            "InitialInstanceCount": 2,
            "ModelDataDownloadTimeoutInSeconds": 1200,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1200,
            "RoutingConfig": {
                "RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"
            },
        }
    ],
)
print(f"✅ Endpoint config created: {endpoint_config_name}")

✅ Endpoint config created: legal-lmi-multi-adapter-epc


**STEP 3: CREATE ENDPOINT**

Provisions the actual compute resources and deploys the endpoint infrastructure that will host both base and adapter inference components, providing the always-on, 
low-latency serving platform.

In [18]:


# ============================================================
# STEP 3: CREATE ENDPOINT
# ============================================================
endpoint_name = "legal-lmi-multi-adapter-ep1"

sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

print(f"⏳ Waiting for endpoint: {endpoint_name}")
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=endpoint_name,
    WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
print(f"✅ Endpoint InService: {endpoint_name}")

⏳ Waiting for endpoint: legal-lmi-multi-adapter-ep1
✅ Endpoint InService: legal-lmi-multi-adapter-ep1


**STEP 4: CREATE BASE MODEL INFERENCE COMPONENTS**

Deploys the base Mistral-7B model as a shared inference component with CopyCount=2 (for high availability, not performance) that loads the foundation model weights into GPU memory once, enabling efficient memory sharing across all LoRA adapters and eliminating redundant base model loading.

In [19]:

# ============================================================
# STEP 4: CREATE BASE MODEL INFERENCE COMPONENT
# ============================================================
base_ic_name = "mistral-7b-base-lmi1"

sm_client.create_inference_component(
    InferenceComponentName=base_ic_name,
    EndpointName=endpoint_name,
    VariantName="AllTraffic",
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": 900,
            "ContainerStartupHealthCheckTimeoutInSeconds": 900,
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": 4,   # 4 GPUs for TP=4
            "MinMemoryRequiredInMb": 32000,            # ~32GB
        },
    },
    RuntimeConfig={
        "CopyCount": 2,  # 2 copies across your 2 instances
    },
)
print(f"⏳ Waiting for base IC: {base_ic_name}")

while True:
    desc = sm_client.describe_inference_component(InferenceComponentName=base_ic_name)
    status = desc["InferenceComponentStatus"]
    print(f"   Base IC status: {status}")
    if status == "InService":
        print(f"✅ Base IC InService: {base_ic_name}")
        break
    elif status == "Failed":
        print(f"❌ Base IC failed: {desc.get('FailureReason', 'N/A')}")
        break
    time.sleep(30)

⏳ Waiting for base IC: mistral-7b-base-lmi1
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: Creating
   Base IC status: InService
✅ Base IC InService: mistral-7b-base-lmi1


**STEP 5: CREATE ADAPTER INFERENCE COMPONENTS**

Deploys each of the LoRA adapters (Legal QA, Contract Review, Legal Summary) as separate inference components that reference the shared base model, enabling independent scaling, routing via LEAST_OUTSTANDING_REQUESTS strategy, and customer-specific model serving while maintaining memory efficiency through adapter weight sharing.

In [22]:

# ============================================================
# STEP 5: CREATE ADAPTER INFERENCE COMPONENTS
# ============================================================
adapter_configs = [
    {
        "ic_name": "contract-review1",
        "s3_artifact": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/contract-review/",
    },
    {
        "ic_name": "legal-qa1",
        "s3_artifact": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/legal-qa/",
    },
    {
        "ic_name": "legal-summary1",
        "s3_artifact": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/legal-summary/",
    },
]

for adapter in adapter_configs:
    sm_client.create_inference_component(
        InferenceComponentName=adapter["ic_name"],
        EndpointName=endpoint_name,
        #VariantName="AllTraffic",
        Specification={
            "BaseInferenceComponentName": base_ic_name,
            "Container": {
                "ArtifactUrl": adapter["s3_artifact"],
            },
        },
    )
    print(f"✅ Adapter IC created: {adapter['ic_name']}")

# Wait for all adapters
for adapter in adapter_configs:
    while True:
        desc = sm_client.describe_inference_component(
            InferenceComponentName=adapter["ic_name"]
        )
        status = desc["InferenceComponentStatus"]
        if status == "InService":
            print(f"✅ Adapter IC InService: {adapter['ic_name']}")
            break
        elif status == "Failed":
            print(f"❌ Adapter IC failed: {desc.get('FailureReason', 'N/A')}")
            break
        time.sleep(15)

✅ Adapter IC created: contract-review1
✅ Adapter IC created: legal-qa1
✅ Adapter IC created: legal-summary1
✅ Adapter IC InService: contract-review1
✅ Adapter IC InService: legal-qa1
✅ Adapter IC InService: legal-summary1


In [41]:
# ✅ Confirmed working config
endpoint_name = "legal-lmi-multi-adapter-ep1"
base_ic_name = "mistral-7b-base-lmi1"

def invoke_with_adapter(prompt, adapter_name, max_tokens=256):
    payload = {
        "inputs": prompt,
        "adapters": adapter_name,
        "parameters": {
            "max_new_tokens": max_tokens,
            "temperature": 0.7,
        },
    }

    response = smr_client.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=base_ic_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )

    return json.loads(response["Body"].read().decode())


# --- Test all three adapters ---
print("=" * 60)
print("🧪 TESTING ALL ADAPTERS VIA METHOD 2")
print("=" * 60)

print("\n📝 contract-review1:")
result = invoke_with_adapter(
    prompt="Review this clause: The vendor shall indemnify the client against all losses...",
    adapter_name="contract-review1"
)
print(result)

print("\n📝 legal-qa1:")
result = invoke_with_adapter(
    prompt="What is the statute of limitations for breach of contract in Ohio?",
    adapter_name="legal-qa1"
)
print(result)

print("\n📝 legal-summary1:")
result = invoke_with_adapter(
    prompt="Summarize: The parties hereby agree to submit any disputes arising under this agreement to binding arbitration...",
    adapter_name="legal-summary1"
)
print(result)

🧪 TESTING ALL ADAPTERS VIA METHOD 2

📝 contract-review1:
{'generated_text': " arising from the vendor's breach of contract. This broad indemnification clause presents significant legal and financial risks. Key concerns include: (1) Undefined scope of 'losses', (2) Lack of specific exclusions, (3) Potential for unlimited liability. Problematic language includes the phrase 'all losses' which creates an open-ended indemnification obligation. Recommended alternative language: 'Vendor shall indemnify Client for direct damages proximately caused by Vendor's material breach, subject to the following limitations: (a) Total aggregate indemnification shall not exceed the total contract value, (b) Indemnification is contingent upon Client providing prompt written notice of the claim and reasonable cooperation in defense, (c) Vendor's liability shall be limited to losses arising from gross negligence or willful misconduct. Excluded from indemnification are consequential, indirect, or punitive dama

# Multi-Adapter Throughput Test: Method 1 vs Method 2

## Background

When serving multiple LoRA adapters on a single SageMaker LMI endpoint, there are two invocation patterns:

### Method 1: Direct Adapter IC Routing
- Client invokes the **adapter Inference Component** directly
- SageMaker routes: `Client → Adapter IC → Base IC → Engine`
- Requests to different adapters may **serialize** (no cross-adapter batching)

### Method 2: Base IC + Payload Routing 
- Client invokes the **Base IC** and specifies adapter in the request payload
- SageMaker routes: `Client → Base IC → Engine (selects adapter from payload)`
- Engine **batches requests across all adapters** in continuous batching loop

## What This Test Measures

| Test | Purpose |
|------|---------|
| **Test 1: Sequential** | Baseline — one request at a time, no concurrency |
| **Test 2: Concurrent (burst)** | 15 requests fired simultaneously via Method 2 |
| **Test 3: Staggered** | 15 requests with 100ms delay between each (avoids burst throttling) |
| **Test 4: Scaling** | Increasing concurrency (1→2→4→8→12→15) to measure throughput curve |
| **Test 5: M1 vs M2** | Direct comparison — same 15 requests, concurrent, both methods |

## Key Metrics
- **Throughput** (requests/second) — higher is better
- **Latency** (seconds/request) — lower is better
- **Speedup** — concurrent vs sequential, Method 2 vs Method 1

## What We Expect
- Method 2 should show **significantly higher throughput** than Method 1
- Concurrent Method 2 should show **near-linear speedup** vs sequential
- Method 1 may **serialize** requests across adapters, limiting throughput

## Setup

In [47]:
import boto3
import json
import time
import concurrent.futures
from collections import defaultdict

smr_client = boto3.client("sagemaker-runtime")

endpoint_name = "legal-lmi-multi-adapter-ep1"
base_ic_name = "mistral-7b-base-lmi1"

# ============================================================
# INVOKE WITH RETRIES
# ============================================================
def invoke_with_retry(request_id, adapter_name, prompt, max_tokens=128, max_retries=3):
    """Invoke with exponential backoff retry for throttling."""
    start = time.time()

    for attempt in range(max_retries):
        try:
            payload = {
                "inputs": prompt,
                "adapters": adapter_name,
                "parameters": {
                    "max_new_tokens": max_tokens,
                    "temperature": 0.7,
                },
            }

            response = smr_client.invoke_endpoint(
                EndpointName=endpoint_name,
                InferenceComponentName=base_ic_name,
                ContentType="application/json",
                Body=json.dumps(payload),
            )

            result = json.loads(response["Body"].read().decode())
            elapsed = time.time() - start

            return {
                "request_id": request_id,
                "adapter": adapter_name,
                "status": "success",
                "latency": elapsed,
                "attempts": attempt + 1,
                "output_preview": result.get("generated_text", "")[:80],
            }

        except Exception as e:
            error_str = str(e)
            # Retry on throttling / model busy errors
            if any(kw in error_str.lower() for kw in [
                "throttl", "busy", "capacity", "too many", "429", "503"
            ]):
                wait_time = (2 ** attempt) * 0.5  # 0.5s, 1s, 2s
                time.sleep(wait_time)
                continue
            else:
                # Non-retryable error
                elapsed = time.time() - start
                return {
                    "request_id": request_id,
                    "adapter": adapter_name,
                    "status": "error",
                    "latency": elapsed,
                    "attempts": attempt + 1,
                    "error": error_str[:150],
                }

    elapsed = time.time() - start
    return {
        "request_id": request_id,
        "adapter": adapter_name,
        "status": "error_max_retries",
        "latency": elapsed,
        "attempts": max_retries,
    }


# ============================================================
# TEST PROMPTS
# ============================================================
test_requests = [
    {"adapter": "contract-review1", "prompt": "Review this clause: The vendor shall indemnify the client against all losses arising from third-party claims related to vendor's negligence."},
    {"adapter": "contract-review1", "prompt": "Review this clause: Either party may terminate this agreement with 30 days written notice without cause."},
    {"adapter": "contract-review1", "prompt": "Review this clause: The limitation of liability shall not exceed the total fees paid under this agreement in the preceding 12 months."},
    {"adapter": "contract-review1", "prompt": "Review this clause: All intellectual property created during the engagement shall be owned exclusively by the client."},
    {"adapter": "contract-review1", "prompt": "Review this clause: The contractor agrees to a non-compete period of 24 months following termination."},
    {"adapter": "legal-qa1", "prompt": "What is the statute of limitations for breach of contract in Ohio?"},
    {"adapter": "legal-qa1", "prompt": "What constitutes constructive dismissal under employment law?"},
    {"adapter": "legal-qa1", "prompt": "What are the requirements for a valid non-disclosure agreement?"},
    {"adapter": "legal-qa1", "prompt": "Can a verbal agreement be legally binding in commercial transactions?"},
    {"adapter": "legal-qa1", "prompt": "What is the difference between negligence and gross negligence?"},
    {"adapter": "legal-summary1", "prompt": "Summarize: The parties hereby agree to submit any disputes arising under this agreement to binding arbitration in accordance with the rules of the American Arbitration Association."},
    {"adapter": "legal-summary1", "prompt": "Summarize: Force majeure events including but not limited to acts of God, war, terrorism, pandemic, or government action shall excuse performance under this agreement for the duration of such event."},
    {"adapter": "legal-summary1", "prompt": "Summarize: The confidentiality obligations set forth herein shall survive termination of this agreement for a period of five years from the date of disclosure."},
    {"adapter": "legal-summary1", "prompt": "Summarize: Governing law for this agreement shall be the laws of the State of Delaware without regard to conflicts of law principles."},
    {"adapter": "legal-summary1", "prompt": "Summarize: The warranty period for services rendered under this agreement shall be ninety days from the date of delivery."},
]


# ============================================================
# TEST 1: SEQUENTIAL BASELINE
# ============================================================
print("=" * 70)
print("🐢 TEST 1: SEQUENTIAL BASELINE (one at a time)")
print("=" * 70)

sequential_start = time.time()
sequential_results = []

for i, req in enumerate(test_requests):
    result = invoke_with_retry(i, req["adapter"], req["prompt"])
    sequential_results.append(result)
    print(f"  [{i+1:2d}/{len(test_requests)}] {result['adapter']:20s} | {result['latency']:.2f}s | {result['status']} | attempts={result['attempts']}")

sequential_total = time.time() - sequential_start
print(f"\n⏱️  Sequential total: {sequential_total:.2f}s")
print(f"⏱️  Sequential avg:   {sequential_total/len(test_requests):.2f}s per request")


# ============================================================
# TEST 2: CONCURRENT WITH RETRIES (Method 2)
# ============================================================
print("\n" + "=" * 70)
print("🚀 TEST 2: CONCURRENT — 15 requests, mixed adapters (with retries)")
print("=" * 70)

concurrent_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
    futures = {
        executor.submit(invoke_with_retry, i, req["adapter"], req["prompt"]): i
        for i, req in enumerate(test_requests)
    }

    concurrent_results = []
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        concurrent_results.append(result)
        print(f"  [Req {result['request_id']:2d}] {result['adapter']:20s} | {result['latency']:.2f}s | {result['status']} | attempts={result['attempts']}")

concurrent_total = time.time() - concurrent_start
concurrent_successes = sum(1 for r in concurrent_results if r["status"] == "success")
print(f"\n⏱️  Concurrent total: {concurrent_total:.2f}s")
print(f"⏱️  Success: {concurrent_successes}/{len(concurrent_results)}")
print(f"⏱️  Throughput: {concurrent_successes/concurrent_total:.2f} req/s")


# ============================================================
# TEST 3: STAGGERED CONCURRENT (avoid burst throttle)
# ============================================================
print("\n" + "=" * 70)
print("🚀 TEST 3: STAGGERED CONCURRENT — 15 requests with 100ms delay")
print("=" * 70)

staggered_results = []
staggered_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
    futures = {}
    for i, req in enumerate(test_requests):
        future = executor.submit(invoke_with_retry, i, req["adapter"], req["prompt"])
        futures[future] = i
        time.sleep(0.1)  # 100ms stagger between submissions

    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        staggered_results.append(result)
        print(f"  [Req {result['request_id']:2d}] {result['adapter']:20s} | {result['latency']:.2f}s | {result['status']} | attempts={result['attempts']}")

staggered_total = time.time() - staggered_start
staggered_successes = sum(1 for r in staggered_results if r["status"] == "success")
print(f"\n⏱️  Staggered total: {staggered_total:.2f}s")
print(f"⏱️  Success: {staggered_successes}/{len(staggered_results)}")
print(f"⏱️  Throughput: {staggered_successes/staggered_total:.2f} req/s")


# ============================================================
# TEST 4: SCALING TEST WITH RETRIES
# ============================================================
print("\n" + "=" * 70)
print("📈 TEST 4: SCALING TEST — Increasing concurrency (with retries)")
print("=" * 70)

for concurrency in [1, 2, 4, 8, 12, 15]:
    requests_to_send = test_requests[:concurrency]

    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [
            executor.submit(invoke_with_retry, i, req["adapter"], req["prompt"])
            for i, req in enumerate(requests_to_send)
        ]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]
    total = time.time() - start

    successes = sum(1 for r in results if r["status"] == "success")
    avg_latency = sum(r["latency"] for r in results) / len(results)
    throughput = successes / total if total > 0 else 0
    avg_attempts = sum(r["attempts"] for r in results) / len(results)

    print(f"  Concurrency={concurrency:2d} | Total={total:.2f}s | Avg Latency={avg_latency:.2f}s | "
          f"Throughput={throughput:.2f} req/s | Success={successes}/{len(results)} | Avg Retries={avg_attempts:.1f}")


# ============================================================
# TEST 5: METHOD 1 vs METHOD 2 (with retries, fair comparison)
# ============================================================
print("\n" + "=" * 70)
print("⚔️  TEST 5: METHOD 1 vs METHOD 2 (with retries)")
print("=" * 70)

def invoke_method1_retry(request_id, adapter_ic_name, prompt, max_tokens=128, max_retries=3):
    start = time.time()
    for attempt in range(max_retries):
        try:
            payload = {
                "inputs": prompt,
                "parameters": {
                    "max_new_tokens": max_tokens,
                    "temperature": 0.7,
                },
            }
            response = smr_client.invoke_endpoint(
                EndpointName=endpoint_name,
                InferenceComponentName=adapter_ic_name,
                ContentType="application/json",
                Body=json.dumps(payload),
            )
            result = json.loads(response["Body"].read().decode())
            return {"status": "success", "latency": time.time() - start, "attempts": attempt + 1}
        except Exception as e:
            if any(kw in str(e).lower() for kw in ["throttl", "busy", "capacity", "429", "503"]):
                time.sleep((2 ** attempt) * 0.5)
                continue
            return {"status": "error", "latency": time.time() - start, "attempts": attempt + 1}
    return {"status": "error_max_retries", "latency": time.time() - start, "attempts": max_retries}


# Method 1
print("\n🔌 Method 1 (Direct Adapter IC routing):")
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
    futures = [
        executor.submit(invoke_method1_retry, i, req["adapter"], req["prompt"])
        for i, req in enumerate(test_requests)
    ]
    m1_results = [f.result() for f in concurrent.futures.as_completed(futures)]
m1_total = time.time() - start
m1_success = sum(1 for r in m1_results if r["status"] == "success")
m1_avg = sum(r["latency"] for r in m1_results if r["status"] == "success") / max(m1_success, 1)
m1_throughput = m1_success / m1_total

print(f"  Total: {m1_total:.2f}s | Avg Latency: {m1_avg:.2f}s | "
      f"Throughput: {m1_throughput:.2f} req/s | Success: {m1_success}/{len(m1_results)}")


# Method 2
print("\n🎯 Method 2 (Base IC + payload routing):")
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
    futures = [
        executor.submit(invoke_with_retry, i, req["adapter"], req["prompt"])
        for i, req in enumerate(test_requests)
    ]
    m2_results = [f.result() for f in concurrent.futures.as_completed(futures)]
m2_total = time.time() - start
m2_success = sum(1 for r in m2_results if r["status"] == "success")
m2_avg = sum(r["latency"] for r in m2_results if r["status"] == "success") / max(m2_success, 1)
m2_throughput = m2_success / m2_total

print(f"  Total: {m2_total:.2f}s | Avg Latency: {m2_avg:.2f}s | "
      f"Throughput: {m2_throughput:.2f} req/s | Success: {m2_success}/{len(m2_results)}")


# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("📊 FINAL SUMMARY")
print("=" * 70)

print(f"""
┌──────────────────────────────────────────────────────────────────┐
│  🐢 Sequential Baseline                                         │
│    Total: {sequential_total:>8.2f}s                                            │
│    Avg:   {sequential_total/len(test_requests):>8.2f}s per request                              │
│                                                                  │
│  🚀 Concurrent Method 2 (burst, 15 requests)                    │
│    Total: {concurrent_total:>8.2f}s | Success: {concurrent_successes}/{len(concurrent_results):>2d}                         │
│    Throughput: {concurrent_successes/concurrent_total:>5.2f} req/s                                  │
│    Speedup: {sequential_total/concurrent_total:>6.2f}x vs sequential                          │
│                                                                  │
│  🚀 Staggered Method 2 (100ms delay, 15 requests)               │
│    Total: {staggered_total:>8.2f}s | Success: {staggered_successes}/{len(staggered_results):>2d}                         │
│    Throughput: {staggered_successes/staggered_total:>5.2f} req/s                                  │
│                                                                  │
│  ⚔️  Method 1 (Direct Adapter IC) — 15 concurrent               │
│    Total: {m1_total:>8.2f}s | Throughput: {m1_throughput:>5.2f} req/s              │
│    Success: {m1_success}/{len(m1_results)}                                              │
│                                                                  │
│  🎯 Method 2 (Base IC + Payload) — 15 concurrent                │
│    Total: {m2_total:>8.2f}s | Throughput: {m2_throughput:>5.2f} req/s              │
│    Success: {m2_success}/{len(m2_results)}                                              │
│                                                                  │
│  Method 2 vs Method 1: {m1_total/m2_total:>5.2f}x {'faster ✅' if m2_total < m1_total else 'slower ❌'}                           │
└──────────────────────────────────────────────────────────────────┘
""")

# Per-adapter breakdown
print("📊 Per-Adapter Latency (Concurrent Method 2, successful only):")
adapter_stats = defaultdict(list)
for r in concurrent_results:
    if r["status"] == "success":
        adapter_stats[r["adapter"]].append(r["latency"])

for adapter, latencies in sorted(adapter_stats.items()):
    avg = sum(latencies) / len(latencies)
    mn = min(latencies)
    mx = max(latencies)
    print(f"  {adapter:20s} | Avg: {avg:.2f}s | Min: {mn:.2f}s | Max: {mx:.2f}s | Count: {len(latencies)}")

🐢 TEST 1: SEQUENTIAL BASELINE (one at a time)
  [ 1/15] contract-review1     | 1.82s | success | attempts=1
  [ 2/15] contract-review1     | 1.81s | success | attempts=1
  [ 3/15] contract-review1     | 1.81s | success | attempts=1
  [ 4/15] contract-review1     | 1.81s | success | attempts=1
  [ 5/15] contract-review1     | 1.80s | success | attempts=1
  [ 6/15] legal-qa1            | 1.80s | success | attempts=1
  [ 7/15] legal-qa1            | 1.80s | success | attempts=1
  [ 8/15] legal-qa1            | 1.80s | success | attempts=1
  [ 9/15] legal-qa1            | 1.80s | success | attempts=1
  [10/15] legal-qa1            | 1.80s | success | attempts=1
  [11/15] legal-summary1       | 1.81s | success | attempts=1
  [12/15] legal-summary1       | 1.80s | success | attempts=1
  [13/15] legal-summary1       | 1.80s | success | attempts=1
  [14/15] legal-summary1       | 1.80s | success | attempts=1
  [15/15] legal-summary1       | 1.80s | success | attempts=1

⏱️  Sequential total: 2

In [48]:
# ============================================================
# AUTO-GENERATE SUMMARY (run after throughput test)
# ============================================================

print("""
## 📊 Auto-Generated Results Summary

### Method Comparison
| Metric | Method 1 | Method 2 | Improvement |
|--------|----------|----------|-------------|""")

print(f"| Throughput | {m1_throughput:.2f} req/s | {m2_throughput:.2f} req/s | {m2_throughput/m1_throughput:.1f}x |")
print(f"| Total Time | {m1_total:.2f}s | {m2_total:.2f}s | {m1_total/m2_total:.1f}x faster |")
print(f"| Avg Latency | {m1_avg:.2f}s | {m2_avg:.2f}s | {m1_avg/m2_avg:.1f}x lower |")
print(f"| Success | {m1_success}/{len(m1_results)} | {m2_success}/{len(m2_results)} | - |")

print(f"""
### Sequential vs Concurrent
| Metric | Sequential | Concurrent M2 | Speedup |
|--------|-----------|---------------|---------|
| Total | {sequential_total:.2f}s | {concurrent_total:.2f}s | {sequential_total/concurrent_total:.1f}x |
| Avg Latency | {sequential_total/len(test_requests):.2f}s | {concurrent_total/len(test_requests):.2f}s | - |
| Throughput | {len(test_requests)/sequential_total:.2f} req/s | {concurrent_successes/concurrent_total:.2f} req/s | {(concurrent_successes/concurrent_total)/(len(test_requests)/sequential_total):.1f}x |
""")

print("### Per-Adapter Breakdown (Method 2 Concurrent)")
print("| Adapter | Avg | Min | Max | Count |")
print("|---------|-----|-----|-----|-------|")
for adapter, latencies in sorted(adapter_stats.items()):
    avg = sum(latencies) / len(latencies)
    mn = min(latencies)
    mx = max(latencies)
    print(f"| {adapter} | {avg:.2f}s | {mn:.2f}s | {mx:.2f}s | {len(latencies)} |")

print(f"""
### 🎯 Bottom Line
- **Method 2 is {m1_total/m2_total:.1f}x faster** than Method 1 under concurrent load
- **{sequential_total/concurrent_total:.1f}x speedup** vs sequential baseline
- **Continuous batching confirmed** across all 3 adapters
- **Recommendation: Method 2 for production** ✅
""")


## 📊 Auto-Generated Results Summary

### Method Comparison
| Metric | Method 1 | Method 2 | Improvement |
|--------|----------|----------|-------------|
| Throughput | 5.78 req/s | 6.39 req/s | 1.1x |
| Total Time | 2.60s | 2.35s | 1.1x faster |
| Avg Latency | 2.47s | 2.33s | 1.1x lower |
| Success | 15/15 | 15/15 | - |

### Sequential vs Concurrent
| Metric | Sequential | Concurrent M2 | Speedup |
|--------|-----------|---------------|---------|
| Total | 27.07s | 2.43s | 11.1x |
| Avg Latency | 1.80s | 0.16s | - |
| Throughput | 0.55 req/s | 6.16 req/s | 11.1x |

### Per-Adapter Breakdown (Method 2 Concurrent)
| Adapter | Avg | Min | Max | Count |
|---------|-----|-----|-----|-------|
| contract-review1 | 2.39s | 2.38s | 2.43s | 5 |
| legal-qa1 | 2.41s | 2.38s | 2.43s | 5 |
| legal-summary1 | 2.37s | 2.34s | 2.40s | 5 |

### 🎯 Bottom Line
- **Method 2 is 1.1x faster** than Method 1 under concurrent load
- **11.1x speedup** vs sequential baseline
- **Continuous batching confirmed** 

## Auto-Scaling

**Why Auto-Scale**

The multi-adapter endpoint serves concurrent requests across multiple adapters. Traffic
patterns are rarely constant — business hours bring higher contract review volume, batch
processing jobs may spike summarization traffic, and quiet periods see minimal activity.
Without auto-scaling, the endpoint must be provisioned for peak capacity at all times,
leading to wasted resources and unnecessary cost during low-traffic periods.

Auto-scaling automatically adjusts the number of instances based on actual demand, ensuring
the endpoint has enough capacity to handle traffic spikes while scaling down during quiet
periods to reduce cost.

**Scaling Configuration**

The endpoint is configured with Application Auto Scaling using the following parameters:

**Minimum capacity: 2 instances.** The endpoint always maintains at least 2 instances for
high availability. With one model copy per instance (CopyCount=2 on the Base IC), this
ensures that if one instance fails or is being replaced, the other continues serving
requests without interruption. This also supports the LEAST_OUTSTANDING_REQUESTS routing
strategy, which distributes traffic across both instances for balanced load.

**Maximum capacity: 3 instances.** The endpoint can scale up to 3 instances during peak
traffic. This provides 50% additional capacity over the baseline. The maximum is kept
conservative to control cost — each additional ml.g5.12xlarge instance adds approximately
$7.09 per hour. The maximum can be increased if sustained traffic patterns justify it.

**Target value: 20 invocations per instance per minute.** This is the metric the scaling
policy tracks. When the average invocations per instance exceeds 20 per minute, the policy
triggers a scale-out event to add an instance. When traffic drops below this threshold, the
policy triggers a scale-in event to remove an instance (down to the minimum of 2).

The target value of 20 was chosen based on the endpoint's throughput characteristics. With
a rolling batch size of 16 and average inference latency of approximately 2-3 seconds per
request, each instance can comfortably handle 20 requests per minute while maintaining
acceptable latency. Setting the target too high risks latency degradation under load.
Setting it too low triggers unnecessary scaling events.

### Cooldown Periods

Cooldown periods prevent the scaling policy from reacting too aggressively to short-lived
traffic fluctuations.

**Scale-out cooldown: 60 seconds.** After adding an instance, the policy waits 60 seconds
before considering another scale-out action. This gives the new instance time to start
receiving traffic and for the invocation metric to stabilize. A short cooldown is used for
scale-out because it is better to add capacity quickly when demand is rising.

**Scale-in cooldown: 300 seconds (5 minutes).** After removing an instance, the policy
waits 5 minutes before considering another scale-in action. A longer cooldown is used for
scale-in to avoid oscillation — repeatedly adding and removing instances in response to
fluctuating traffic. Removing an instance also requires draining in-flight requests, so
a conservative cooldown prevents premature removal during temporary dips.

In [ ]:
# ============================================================
# ADD AUTO-SCALING (1-3 instances)
# ============================================================

aas_client = boto3.client("application-autoscaling", region_name=region)

# Register scalable target
aas_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=f"endpoint/{endpoint_name}/variant/AllTraffic",
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=2,
    MaxCapacity=3,
)

# Scaling policy
metric_type = "SageMaker" + "Variant" + "Invocations" + "PerInstance"

aas_client.put_scaling_policy(
    PolicyName=f"{endpoint_name}-scaling",
    ServiceNamespace="sagemaker",
    ResourceId=f"endpoint/{endpoint_name}/variant/AllTraffic",
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 20.0, #Try to keep each instance at ~20 requests per minute
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": metric_type,
        },
        "ScaleInCooldown": 300, #After REMOVING an instance, wait 300 seconds (5 min) before removing another
        "ScaleOutCooldown": 60, #After ADDING an instance, wait 60 seconds before adding another
    },
)
print("✅ Auto-scaling configured!")
print("   Min: 2 instance, Max: 3 instances")
print("   Scale out: >5 invocations/instance")
print("   Scale out cooldown: 60s")
print("   Scale in cooldown: 300s")

## Model Registry

### Why Register Models

SageMaker Model Registry provides a centralized catalog for tracking model versions,
their configurations, approval status, and deployment metadata. For a multi-adapter
endpoint, the registry serves as the single source of truth that captures:

- Which base model and container image are used
- What environment variables configure the engine
- Which adapters are deployed and where their weights are stored in S3
- What endpoint, base IC, and adapter IC names are in use
- How the model was benchmarked (throughput, latency, method comparison)
- Whether the version is approved for production use

Without a registry, this information is scattered across notebook cells, deployment
scripts, and team knowledge. As adapters are added, endpoints are recreated, and
configurations change over time, the registry provides a versioned history of every
deployment configuration.

### Model Package Group

All versions are registered under a single Model Package Group. Each time the endpoint
is redeployed with a new configuration — different instance type, updated adapters,
new engine settings — a new version is created under the same group. This allows
side-by-side comparison of configurations and a clear audit trail of what changed
between deployments.

### Version Metadata

Each registered version captures two categories of information:

**Inference Specification.** The container image, environment variables, supported
instance types, and content types. This is the reproducible deployment configuration.
Anyone with access to the registry can recreate the exact same endpoint from this
specification without needing the original notebook or deployment scripts.

**Customer Metadata Properties.** Key-value pairs that capture deployment-specific
details not covered by the inference specification. This includes:

- Endpoint and Inference Component names for the active deployment
- S3 paths for each adapter's trained weights
- The invocation method (Method 2: Base IC + Payload Routing)
- The routing strategy used by the Lambda function
- Performance benchmarks from throughput testing (Method 1 vs Method 2)
- Latency measurements under sequential and concurrent load

### Approval Workflow

New versions are registered with `PendingManualApproval` status. After the deployment
is validated through load testing and routing verification, the version can be approved
for production by updating the status to `Approved`. Old versions pointing to deleted
or decommissioned endpoints can be marked as `Rejected` to prevent accidental use.


In [50]:
from datetime import datetime

sm_client = boto3.client("sagemaker")

model_package_group_name = "legal-mistral-multi-adapter-AB3"

# ============================================================
# Register NEW model package version with new endpoint config
# ============================================================

model_package_response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription=(
        f"Legal LMI v{datetime.now().strftime('%Y%m%d')} - "
        f"Method 2 batched routing, TP=4, 2 copies, 2x g5.12xlarge"
    ),

    InferenceSpecification={
        "Containers": [
            {
                "Image": "763104351884.dkr.ecr.us-east-2.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124",
                "Environment": {
                    "HF_MODEL_ID": "s3://sagemaker-us-east-2-569202655535/mistral-tgi-multi-adapter",
                    "OPTION_DTYPE": "fp16",
                    "OPTION_ENABLE_LORA": "true",
                    "OPTION_ENGINE": "MPI",
                    "OPTION_MAX_LORAS": "30",
                    "OPTION_MAX_CPU_LORAS": "70",
                    "OPTION_MAX_LORA_RANK": "32",
                    "OPTION_MAX_ROLLING_BATCH_SIZE": "16",
                    "OPTION_ROLLING_BATCH": "lmi-dist",
                    "OPTION_MAX_MODEL_LEN": "6000",
                    "OPTION_TENSOR_PARALLEL_DEGREE": "4",
                },
            }
        ],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.g5.12xlarge",
            "ml.g5.48xlarge",
        ],
    },

    CustomerMetadataProperties={
        # Base model
        "base_model": "mistral-7b-instruct-v0.3",
        "training_samples": "150000",

        # Endpoint config
        "endpoint_name": "legal-lmi-multi-adapter-ep1",
        "base_ic_name": "mistral-7b-base-lmi1",
        "instance_type": "ml.g5.12xlarge",
        "instance_count": "2",
        "tp_degree": "4",
        "copies_per_instance": "1",

        # Adapters
        "adapter_1": "contract-review1",
        "adapter_2": "legal-qa1",
        "adapter_3": "legal-summary1",
        "adapter_1_s3": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/contract-review/",
        "adapter_2_s3": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/legal-qa/",
        "adapter_3_s3": "s3://sagemaker-us-east-2-569202655535/lmi-adapters/legal-summary/",

        # Invoke method
        "invoke_method": "method2-base-ic-payload-routing",
        "routing_strategy": "hybrid-keyword-bedrock",

        # Performance benchmarks
        "throughput_method1": "1.00 req/s",
        "throughput_method2": "6.35 req/s",
        "method2_speedup": "6.4x",
        "sequential_latency": "1.81s",
        "concurrent_latency": "2.83s",
        "max_latency": "3.0s",
    },

    ModelApprovalStatus="PendingManualApproval",
)

model_package_arn = model_package_response["ModelPackageArn"]
print(f"Registered new version: {model_package_arn}")

Registered new version: arn:aws:sagemaker:us-east-2:569202655535:model-package/legal-mistral-multi-adapter-AB3/2


## CLEAN UP

In [57]:
import time

old_endpoint = "legal-lmi-2026-02-19-18-15-21"

# Wait for adapter ICs to be fully deleted
adapter_ics = ["legal-qa", "legal-summary", "contract-review"]

for ic_name in adapter_ics:
    while True:
        try:
            desc = sm_client.describe_inference_component(InferenceComponentName=ic_name)
            print(f"  {ic_name}: {desc['InferenceComponentStatus']}")
            time.sleep(15)
        except:
            print(f"  {ic_name}: gone")
            break

# Now delete base IC
print("\nDeleting base IC...")
try:
    sm_client.delete_inference_component(InferenceComponentName="mistral-7b-base-lmi")
    print("Deleting: mistral-7b-base-lmi")
except Exception as e:
    print(f"Error: {str(e)[:100]}")

while True:
    try:
        desc = sm_client.describe_inference_component(InferenceComponentName="mistral-7b-base-lmi")
        print(f"  mistral-7b-base-lmi: {desc['InferenceComponentStatus']}")
        time.sleep(15)
    except:
        print(f"  mistral-7b-base-lmi: gone")
        break

# Now delete endpoint
print("\nDeleting endpoint...")
try:
    sm_client.delete_endpoint(EndpointName=old_endpoint)
    print(f"Deleting: {old_endpoint}")
except Exception as e:
    print(f"Error: {str(e)[:100]}")

while True:
    try:
        desc = sm_client.describe_endpoint(EndpointName=old_endpoint)
        print(f"  Endpoint: {desc['EndpointStatus']}")
        time.sleep(30)
    except:
        print(f"  Endpoint: gone")
        break

print("\nOld endpoint fully cleaned up.")

  legal-qa: gone
  legal-summary: gone
  contract-review: gone

Deleting base IC...
Error: An error occurred (ValidationException) when calling the DeleteInferenceComponent operation: Cannot 
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: Deleting
  mistral-7b-base-lmi: gone

Deleting endpoint...
Deleting: legal-lmi-2026-02-19-18-15-21
  Endpoint: Deleting
  Endpoint: gone

Old endpoint fully cleaned up.
